# MLP for Binary Classification on Network Traffic
This notebook trains a Multi-Layer Perceptron (MLP) to classify network flows as benign or malicious using the CIC-DDoS2019 dataset. The input data consists of 100 packets per flow, each with 20 features.

In [ ]:
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from util_functions import load_dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 36.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 80.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.1/111.1 kB 14.3 MB/s eta 0:00:00


## Load Dataset
The dataset is loaded from the DOS2019 folder. Each flow is represented as a 100x20 array.

In [ ]:
DATASET_FOLDER = '../10-CNNs/DOS2019'
EPOCHS = 100

X_train, y_train = load_dataset(DATASET_FOLDER + '/*-train.hdf5')
X_val, y_val = load_dataset(DATASET_FOLDER + '/*-val.hdf5')
print('Train shape:', X_train.shape, y_train.shape)
print('Val shape:', X_val.shape, y_val.shape)

## Build and Train MLP Model
The MLP model flattens the input and uses two dense layers for binary classification.

In [ ]:
mlp_model = Sequential()
mlp_model.add(Flatten(input_shape=X_train.shape[1:]))
mlp_model.add(Dense(100, activation='relu'))
mlp_model.add(Dense(1, activation='sigmoid'))
mlp_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
mlp_model.summary()

In [ ]:
start_time = time.time()
history = mlp_model.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_val, y_val), batch_size=16, verbose=1)
stop_time = time.time()
print('Training time/epoch (sec):', (stop_time-start_time)/EPOCHS)

## Plot Training History

In [ ]:
def plot_history(history):
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')
    plt.show()
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    plt.show()

plot_history(history)

## Evaluate Model Performance

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix
y_pred = (mlp_model.predict(X_val) > 0.5).astype(int)
f1 = f1_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
print('F1 Score:', f1)
print('False Negative Rate:', fnr)
print('False Positive Rate:', fpr)

## Save and Load Model

In [ ]:
mlp_model.save('mlp_binary_classification.h5')
loaded_model = tf.keras.models.load_model('mlp_binary_classification.h5')
loaded_model.summary()